# Actually when a graph gets executed, its state gets erased as soon it gets executed, so we need to make sure that whenever we would access a graph, we are accessing it from the last state.

# Persistance is used to save the state of the workflow, so that whenever u access the state, you may start from where u left off.

In [1]:
# Formal Definition

# Persistance in langgraph refers to the ability to save and restore the state of a workflow over time.

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [12]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'cricket'}, config=config2)

{'topic': 'cricket',
 'joke': 'Why did the cricket team go to the bakery? \n\nBecause they heard they could get a good bouncer there!',
 'explanation': 'This joke plays on the double meaning of the term "bouncer." In cricket, a "bouncer" is a fast, intimidating delivery bowled at the batsman. However, in this joke, "bouncer" also refers to a type of bread that is sold at a bakery. The cricket team went to the bakery because they heard they could get a good "bouncer" there, but they were referring to the bread rather than the cricket delivery. This unexpected twist adds humor to the punchline of the joke.'}

In [14]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket team go to the bakery? \n\nBecause they heard they could get a good bouncer there!', 'explanation': 'This joke plays on the double meaning of the term "bouncer." In cricket, a "bouncer" is a fast, intimidating delivery bowled at the batsman. However, in this joke, "bouncer" also refers to a type of bread that is sold at a bakery. The cricket team went to the bakery because they heard they could get a good "bouncer" there, but they were referring to the bread rather than the cricket delivery. This unexpected twist adds humor to the punchline of the joke.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b47bd-b6e8-688b-8002-60572b28d5da'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-19T22:45:48.062733+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1b47bd-9976-60ba-8001-33187de74a54'}}, t

In [15]:
# Against each config there is a separate state.

# Benefits of persistance

# 01- Short term memory
# 02- Human in the loop
# 03- Fault Tolerance
# 04- Time Travel